# Ch 30. ZeRO와 FSDP — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 포함합니다.


## 문제 1

**문제**: ZeRO Stage 0/1/2/3의 메모리를 70B 모델, N=16에 대해 계산하라.

### 풀이

혼합정밀도 Adam의 단순 모델은 FP16 parameter 2P, gradient 2P, FP32 master+moments 12P로 총 16P byte다. Stage 1/2/3은 차례로 optimizer, gradient, parameter까지 $N$등분한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
P=70; N=16
mem={'stage0':16*P,'stage1':4*P+12*P/N,'stage2':4*P+(2+12)*P/N,'stage3':16*P/N}
assert mem['stage3']==70.; mem


## 문제 2

**문제**: FSDP가 순전파 전 All-Gather를 하는 이유를 설명하라.

### 풀이

각 rank는 parameter shard만 보유하지만 dense layer 계산에는 전체 weight가 필요하다. 순전파 직전 All-Gather로 해당 layer의 full parameter를 잠시 재구성하고 사용 뒤 reshard한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
P,N=1000,4; local=P//N; gathered=local*N
assert gathered==P
{'local_shard':local,'required_for_layer_compute':gathered}


## 문제 3

**문제**: ZeRO-3의 통신 오버헤드를 DDP와 비교하라.

### 풀이

DDP ring all-reduce는 gradient에 대략 $2(N-1)P/N$ 통신을 쓴다. ZeRO-3은 gradient reduce-scatter 외에도 forward와 backward parameter all-gather가 있어 통신 event가 더 많고 bucket/prefetch overlap이 중요하다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
P=1.; ddp=2*(1-1/8)*P; zero3_forward=P; zero3_backward=2*P
assert zero3_forward+zero3_backward>ddp
{'DDP_ring_units':ddp,'ZeRO3_approx_units':zero3_forward+zero3_backward}


## 문제 4

**문제**: CPU Offloading이 속도를 느리게 하는 이유를 설명하라.

### 풀이

offload는 GPU HBM 대신 느린 CPU DRAM에 상태를 두고 PCIe/NVLink 전송을 critical path에 추가한다. 전송 하한은 bytes/bandwidth이며 작은 kernel과 동기화가 latency를 더한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
bytes_to_move=8e9; pcie_bandwidth=32e9; transfer_seconds=bytes_to_move/pcie_bandwidth
assert transfer_seconds>.2; {'one_way_transfer_lower_bound_seconds':transfer_seconds}


## 문제 5

**문제**: 70B 모델을 학습하려면 ZeRO-3 + 몇 개 A100 80GB가 필요한지 계산하라.

### 풀이

70B의 16P 상태는 약 1120 GB다. 80 GB의 15%를 activation·fragmentation headroom으로 남기면 유효 68 GB이므로 이상적 하한은 $\lceil1120/68\rceil=17$대이며 실제 수는 activation checkpoint와 sequence 길이로 검증한다.

아래 코드는 고정된 난수 시드의 소규모 CPU 실험이다. 절대 시간이나 대규모 품질 수치를 주장하지 않고, 비교해야 할 수학적 양과 불변식을 검증한다.


In [ ]:
import math
parameters_billion=70; bytes_per_parameter=16; gpu_capacity_gb=80; headroom=.15
total_gb=parameters_billion*bytes_per_parameter
usable_per_gpu=gpu_capacity_gb*(1-headroom)
minimum=math.ceil(total_gb/usable_per_gpu)
feasible=lambda n: n*usable_per_gpu>=total_gb
enumerated=next(n for n in range(1,1000) if feasible(n))
assert minimum==enumerated and feasible(minimum) and not feasible(minimum-1)
{'model_state_gb':total_gb,'usable_gb_per_gpu':usable_per_gpu,'ideal_minimum':minimum,
 'boundary_check':{'previous_capacity_gb':(minimum-1)*usable_per_gpu,'minimum_capacity_gb':minimum*usable_per_gpu},
 'note':'ideal model-state bound only; measured activations and fragmentation can require more GPUs'}
